# Shoof layer data (step 1 of 2)
Makes training data for Shoof's Magic layers: thousands of designs built by Shoof itself, each saved with its exact layers (text with font/size/alignment, shapes, photos, shadows as their own layers).

**Settings (right panel):** Internet **On**, Accelerator **None** (CPU is enough, saves GPU hours). Then **Save Version → Save & Run All**. Takes about 2–4 hours. The result appears in the Output tab as `ld-*.tar` files.

In [ ]:
N_DESIGNS = 40000   # more = better model, longer run
SIZE = 512
!git clone -q --depth 1 https://github.com/shahrosekhalid96-debug/shoof /kaggle/temp/shoof
!pip install -q playwright pillow
!playwright install --with-deps chromium > /dev/null 2>&1; echo browser ready

In [ ]:
# free CC0 / public-domain photos to put inside the designs (sources listed in SOURCES.tsv)
!python /kaggle/temp/shoof/tools/layerdata/photos.py --out /kaggle/temp/photos --n 800

In [ ]:
import os
W = os.cpu_count()
!python /kaggle/temp/shoof/tools/layerdata/gen.py --out /kaggle/temp/ld --n {N_DESIGNS} --size {SIZE} --workers {W} --photos /kaggle/temp/photos --sheet /kaggle/working/sample-sheet.png

In [ ]:
# pack into a few tar files (Kaggle handles a few big files better than 400,000 small ones)
import os, tarfile
ds = sorted(os.listdir('/kaggle/temp/ld'), key=lambda x: int(x) if x.isdigit() else -1)
ds = [d for d in ds if d.isdigit()]
for i in range(0, len(ds), 5000):
    with tarfile.open(f'/kaggle/working/ld-{i//5000:02d}.tar', 'w') as t:
        for d in ds[i:i+5000]:
            t.add(f'/kaggle/temp/ld/{d}', arcname=d)
!cp /kaggle/temp/photos/SOURCES.tsv /kaggle/working/photo-sources.tsv
!ls -la /kaggle/working | head -20